# Ch 04 · NVIDIA Warp and How a GPU Runs Your Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/04-warp-and-the-gpu.ipynb)

Companion notebook for the chapter page `chapters/04-warp-and-the-gpu.html` ([page](../chapters/04-warp-and-the-gpu.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## 1. Devices and 2. arrays
Warp arrays have a dtype, a shape and a device. Vector dtypes (`wp.vec3`, `wp.quat`, `wp.mat33`, `wp.spatial_vector`) count as one element in the Warp shape and add trailing dims in NumPy. `.numpy()` on a GPU array waits for the GPU and copies; on the CPU device it is a view.

In [ ]:
# Example 4.1 - ex1_arrays.py
"""Example 4.1 - wp.array basics: dtype, shape, device, NumPy views of vector dtypes.

Creates float, vec3, quat, mat33 and spatial_vector arrays and prints their Warp shape next to
the shape of their NumPy copy (vector dtypes add trailing NumPy dims). Also shows the copy and
fill helpers MJWarp code uses all the time.
Run: python ex1_arrays.py
"""
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

np.set_printoptions(precision=4, suppress=True)

device = wp.get_device()            # "cuda:0" on Colab with a GPU, "cpu" otherwise
print("default device:", device, "| is_cuda:", device.is_cuda)

# 1. scalar arrays: zeros / ones / empty / from NumPy
a = wp.zeros(4, dtype=float)                       # float means float32 in Warp
b = wp.array(np.arange(4, dtype=np.float32))       # copies host -> device
print("a:", a.shape, a.dtype.__name__, a.device, "| b.numpy():", b.numpy())

# 2. vector and matrix dtypes: the Warp shape counts elements, NumPy adds the trailing dims
arrays = {
    "vec3": wp.zeros(4, dtype=wp.vec3),
    "quat": wp.zeros(4, dtype=wp.quat),
    "mat33": wp.zeros(4, dtype=wp.mat33),
    "spatial_vector": wp.zeros(4, dtype=wp.spatial_vector),
    "vec3 2d (nworld=2, nbody=3)": wp.zeros((2, 3), dtype=wp.vec3),
}
for name, arr in arrays.items():
    print(f"{name:28s} warp shape {str(arr.shape):8s} numpy shape {arr.numpy().shape}")

# 3. NumPy (N, 3) -> vec3 array: the last NumPy dim is absorbed into the dtype
pts = wp.array(np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32), dtype=wp.vec3)
print("vec3 from (2, 3) numpy:", pts.shape, "first element:", pts.numpy()[0])

# 4. in-place helpers: assign (copy from host), fill_, zero_, wp.copy (device -> device)
a.assign(np.array([10, 20, 30, 40], dtype=np.float32))
c = wp.empty_like(a)                               # empty: contents undefined until written
wp.copy(c, a)
a.fill_(7.0)
print("after assign+copy+fill_: a =", a.numpy(), "c =", c.numpy())
c.zero_()
print("after c.zero_():", c.numpy())

# 5. .numpy() on a CUDA array waits for the GPU and copies to the host; on the CPU device it
#    returns a view that shares memory with the Warp array. Never rely on either side effect.
host = b.numpy()
host[0] = 99.0
kind = "copy (device unchanged)" if b.device.is_cuda else "view (CPU arrays share memory)"
print(f"b after editing its .numpy() result: {b.numpy()} -> {kind}")

# 6. choose the device explicitly (cpu always exists)
with wp.ScopedDevice("cpu"):
    on_cpu = wp.ones(3, dtype=wp.vec3)
print("ScopedDevice('cpu'):", on_cpu.device, on_cpu.numpy().shape)

## 3. Kernels and threads
A `@wp.kernel` runs once per index of `dim`; `wp.tid()` returns the index. `dim=0` is a safe no-op.

In [ ]:
# Example 4.2 - ex2_saxpy.py
"""Example 4.2 - first kernel: y = a * x + y on 1M elements (SAXPY).

One thread per element. Compares the result with NumPy.
Run: python ex2_saxpy.py
"""
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

np.set_printoptions(precision=4, suppress=True)

N = 1 << 20                                        # 1,048,576 elements (same on CPU and GPU)


@wp.kernel
def saxpy(a: float, x: wp.array[float], y: wp.array[float]):
    i = wp.tid()                                   # this thread's index, 0 .. N-1
    y[i] = a * x[i] + y[i]


rng = np.random.default_rng(0)
x_np = rng.standard_normal(N).astype(np.float32)
y_np = rng.standard_normal(N).astype(np.float32)

x = wp.array(x_np)
y = wp.array(y_np)
wp.launch(saxpy, dim=N, inputs=[2.0, x], outputs=[y])   # first launch compiles (JIT)

y_out = y.numpy()                                  # .numpy() waits for the device, then copies
ref = 2.0 * x_np + y_np
print("device:", wp.get_device(), "| N =", N)
print("first 3 values:", y_out[:3])
print("NumPy reference:", ref[:3])
print("max abs error:", float(np.max(np.abs(y_out - ref))))

wp.launch(saxpy, dim=0, inputs=[2.0, x], outputs=[y])   # dim=0 is a safe no-op
print("dim=0 launch ok, y unchanged:", bool(np.array_equal(y.numpy(), y_out)))

A 2-D launch `dim=(nworld, ndof)` returns two indices; the last one varies fastest. On the CPU device threads run in row-major order. On a GPU the order is not guaranteed; with only 24 threads (a single warp) it may still come out in order, so do not rely on either.

In [ ]:
# Example 4.3 - ex3_grid_2d.py
"""Example 4.3 - a 2D launch over (nworld, ndof): wp.tid() returns two indices.

Each thread writes worldid * 100 + dofid, the same shape of launch MJWarp uses for
dim=(d.nworld, m.nv). A second kernel records the order in which threads ran.
Run: python ex3_grid_2d.py [--dump-json PATH]
  --dump-json PATH writes the grid and the CPU run order for the page's thread-grid figure.
"""
import argparse
import json

import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD, NDOF = 4, 6


@wp.kernel
def label(out: wp.array2d[int]):
    worldid, dofid = wp.tid()                      # 2D launch -> two indices
    out[worldid, dofid] = worldid * 100 + dofid


@wp.kernel
def run_order(counter: wp.array[int], order: wp.array2d[int]):
    worldid, dofid = wp.tid()
    order[worldid, dofid] = wp.atomic_add(counter, 0, 1)   # ticket = when this thread ran


grid = wp.zeros((NWORLD, NDOF), dtype=int)
wp.launch(label, dim=(NWORLD, NDOF), outputs=[grid])
print("launch dim:", (NWORLD, NDOF), "->", NWORLD * NDOF, "threads")
print(grid.numpy())

counter = wp.zeros(1, dtype=int)
order = wp.zeros((NWORLD, NDOF), dtype=int)
wp.launch(run_order, dim=(NWORLD, NDOF), inputs=[counter], outputs=[order])
print("run order on", wp.get_device(), "(on a GPU the order is not defined):")
print(order.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "ndof": NDOF, "grid": grid.numpy().tolist(),
                   "order": order.numpy().tolist(), "device": str(wp.get_device())},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 4. How a GPU runs your kernel
Warps of 32 lanes run in lockstep. **Divergence:** when lanes of one warp take different branches, the warp runs both sides. On a GPU the interleaved variant is expected to be the slowest (check your own numbers below); on the CPU device the three times are close.

In [ ]:
# Example 4.6 - ex6_divergence.py
"""Example 4.6 - branch divergence: 8 cases selected by tid % 8 vs grouped vs 8 launches.

Every element runs one of 8 different loops. Three ways to schedule the same work:
  interleaved  one launch, branch = i % 8      -> the 32 lanes of a GPU warp disagree (divergent)
  grouped      one launch, branch = i // M     -> all lanes of a warp agree (no divergence)
  specialized  8 launches, one branch each     -> no divergence, 8 launch overheads
On a GPU the interleaved kernel is expected to be the slowest; run it and compare your own numbers.
On Warp's CPU device threads run one after another, there are no lanes, so the three times are
close: divergence is a GPU effect.
Run: python ex6_divergence.py [--dump-json PATH]
"""
import argparse
import json
import time

import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
N = (1 << 22) if ON_GPU else (1 << 16)            # elements; multiple of 8 * 32
M = N // 8                                         # elements per branch
ITERS = 64


@wp.func
def work(branch: int, x: float) -> float:
    r = x
    if branch == 0:
        for _ in range(ITERS):
            r = wp.sin(r) + 0.5
    elif branch == 1:
        for _ in range(ITERS):
            r = wp.cos(r) * 0.9
    elif branch == 2:
        for _ in range(ITERS):
            r = wp.sqrt(r * r + 1.0) - 0.9
    elif branch == 3:
        for _ in range(ITERS):
            r = wp.exp(-r * r)
    elif branch == 4:
        for _ in range(ITERS):
            r = wp.tanh(r) + 0.25
    elif branch == 5:
        for _ in range(ITERS):
            r = 1.0 / (1.0 + r * r)
    elif branch == 6:
        for _ in range(ITERS):
            r = wp.log(1.0 + r * r)
    else:
        for _ in range(ITERS):
            r = wp.atan(r) + 0.1
    return r


@wp.kernel
def interleaved(x: wp.array[float], out: wp.array[float]):
    i = wp.tid()
    out[i] = work(i % 8, x[i])                     # neighbours take different branches


@wp.kernel
def grouped(m: int, x: wp.array[float], out: wp.array[float]):
    i = wp.tid()
    out[i] = work(i // m, x[i])                    # a whole warp shares one branch


@wp.kernel
def one_branch(branch: int, m: int, x: wp.array[float], out: wp.array[float]):
    j = wp.tid()
    i = branch * m + j
    out[i] = work(branch, x[i])                    # every thread of this launch: same branch


x_np = np.random.default_rng(0).random(N, dtype=np.float32)   # grouped order: branch-major
x = wp.array(x_np)
# the same values in interleaved order: element 8 * j + c holds x_np[c * M + j]
x_inter = wp.array(x_np.reshape(8, M).T.copy().reshape(N))
out_a = wp.zeros(N, dtype=float)
out_b = wp.zeros(N, dtype=float)
out_c = wp.zeros(N, dtype=float)


def run_interleaved():
    wp.launch(interleaved, dim=N, inputs=[x_inter], outputs=[out_a])


def run_grouped():
    wp.launch(grouped, dim=N, inputs=[M, x], outputs=[out_b])


def run_specialized():
    for branch in range(8):
        wp.launch(one_branch, dim=M, inputs=[branch, M, x], outputs=[out_c])


def best_ms(fn, reps=5):
    fn()                                           # warm-up: JIT compile + first run
    wp.synchronize()
    best = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        wp.synchronize()                           # wait for the device before reading the clock
        best = min(best, time.perf_counter() - t0)
    return 1e3 * best


times = {"interleaved": best_ms(run_interleaved), "grouped": best_ms(run_grouped),
         "specialized": best_ms(run_specialized)}

# all three compute the same numbers (interleaved stores them in interleaved order)
a_back = out_a.numpy().reshape(M, 8).T.reshape(N)
print(f"device: {wp.get_device()} | N = {N} | {ITERS} iterations per element")
for k, v in times.items():
    print(f"  {k:12s} {v:8.2f} ms   ({v / times['grouped']:.2f}x grouped)")
print("same results in all three:",
      bool(np.array_equal(out_b.numpy(), out_c.numpy()) and np.array_equal(a_back, out_b.numpy())))
if not ON_GPU:
    print("CPU device: threads run in a loop, no 32-lane warps, so no divergence penalty.")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "n": N, "iters": ITERS,
                   "ms": {k: round(v, 2) for k, v in times.items()}}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

**Coalescing:** neighbouring threads should read neighbouring addresses. The strided kernel walks the row index fastest.

In [ ]:
# Example 4.7 - ex7_coalescing.py
"""Example 4.7 - memory coalescing: which index do consecutive threads walk?

Adds two (nworld, n) arrays. Both kernels do identical work; only the mapping from thread
index to array element differs:
  coalesced  w, j = wp.tid() with dim=(nworld, n): neighbouring threads read neighbouring
             addresses (the contiguous last axis) -> a GPU warp needs few memory transactions
  strided    j, w = wp.tid() with dim=(n, nworld): neighbouring threads are a whole row
             (n * 4 bytes) apart -> one transaction per lane
This is why MJWarp puts nworld first and launches dim=(nworld, n_elements).
On the CPU device threads run in order, so the strided kernel jumps n * 4 bytes per thread
and loses the CPU cache instead: it is slower too, for a different reason.
Run: python ex7_coalescing.py [--dump-json PATH]
"""
import argparse
import json
import time

import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD, N = (8192, 4096) if ON_GPU else (2048, 2048)


@wp.kernel
def add_coalesced(a: wp.array2d[float], b: wp.array2d[float], c: wp.array2d[float]):
    w, j = wp.tid()                                # last launch axis = contiguous array axis
    c[w, j] = a[w, j] + b[w, j]


@wp.kernel
def add_strided(a: wp.array2d[float], b: wp.array2d[float], c: wp.array2d[float]):
    j, w = wp.tid()                                # last launch axis = the row index
    c[w, j] = a[w, j] + b[w, j]


rng = np.random.default_rng(0)
a = wp.array(rng.random((NWORLD, N), dtype=np.float32))
b = wp.array(rng.random((NWORLD, N), dtype=np.float32))
c1 = wp.zeros((NWORLD, N), dtype=float)
c2 = wp.zeros((NWORLD, N), dtype=float)


def best_ms(fn, reps=5):
    fn()                                           # warm-up (JIT)
    wp.synchronize()
    best = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        wp.synchronize()
        best = min(best, time.perf_counter() - t0)
    return 1e3 * best


t_co = best_ms(lambda: wp.launch(add_coalesced, dim=(NWORLD, N), inputs=[a, b], outputs=[c1]))
t_st = best_ms(lambda: wp.launch(add_strided, dim=(N, NWORLD), inputs=[a, b], outputs=[c2]))
gbytes = 3 * NWORLD * N * 4 / 1e9                  # 2 reads + 1 write of float32

print(f"device: {wp.get_device()} | array ({NWORLD}, {N}) float32, {gbytes:.3f} GB moved")
print(f"  coalesced {t_co:8.2f} ms  {gbytes / (t_co / 1e3):7.1f} GB/s")
print(f"  strided   {t_st:8.2f} ms  {gbytes / (t_st / 1e3):7.1f} GB/s"
      f"   ({t_st / t_co:.1f}x slower)")
print("same result:", bool(np.array_equal(c1.numpy(), c2.numpy())))
print("thread k reads address offset (floats): coalesced k, strided k *", N)
if not ON_GPU:
    print("CPU device: the gap comes from CPU caches, not from GPU memory transactions.")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "shape": [NWORLD, N],
                   "ms": {"coalesced": round(t_co, 2), "strided": round(t_st, 2)}},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

### GPU vs CPU
The next cells write three of the scripts to disk and run each on the GPU and on Warp's CPU device (a subprocess with `CUDA_VISIBLE_DEVICES` empty). Numbers are yours; nothing is hard-coded. **The sizes differ per device**: each script picks its budget from the device it sees (ex6: N = 4,194,304 on the GPU vs 65,536 on the CPU; ex7: (8192, 4096) vs (2048, 2048); only ex2 uses the same N = 1,048,576 on both), and each run prints its size. So apart from ex2, do not compare milliseconds across devices; compare the ratios *within* each device (interleaved vs grouped, strided vs coalesced).

In [ ]:
%%writefile ex2_saxpy.py
"""Example 4.2 - first kernel: y = a * x + y on 1M elements (SAXPY).

One thread per element. Compares the result with NumPy.
Run: python ex2_saxpy.py
"""
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

np.set_printoptions(precision=4, suppress=True)

N = 1 << 20                                        # 1,048,576 elements (same on CPU and GPU)


@wp.kernel
def saxpy(a: float, x: wp.array[float], y: wp.array[float]):
    i = wp.tid()                                   # this thread's index, 0 .. N-1
    y[i] = a * x[i] + y[i]


rng = np.random.default_rng(0)
x_np = rng.standard_normal(N).astype(np.float32)
y_np = rng.standard_normal(N).astype(np.float32)

x = wp.array(x_np)
y = wp.array(y_np)
wp.launch(saxpy, dim=N, inputs=[2.0, x], outputs=[y])   # first launch compiles (JIT)

y_out = y.numpy()                                  # .numpy() waits for the device, then copies
ref = 2.0 * x_np + y_np
print("device:", wp.get_device(), "| N =", N)
print("first 3 values:", y_out[:3])
print("NumPy reference:", ref[:3])
print("max abs error:", float(np.max(np.abs(y_out - ref))))

wp.launch(saxpy, dim=0, inputs=[2.0, x], outputs=[y])   # dim=0 is a safe no-op
print("dim=0 launch ok, y unchanged:", bool(np.array_equal(y.numpy(), y_out)))

In [ ]:
%%writefile ex6_divergence.py
"""Example 4.6 - branch divergence: 8 cases selected by tid % 8 vs grouped vs 8 launches.

Every element runs one of 8 different loops. Three ways to schedule the same work:
  interleaved  one launch, branch = i % 8      -> the 32 lanes of a GPU warp disagree (divergent)
  grouped      one launch, branch = i // M     -> all lanes of a warp agree (no divergence)
  specialized  8 launches, one branch each     -> no divergence, 8 launch overheads
On a GPU the interleaved kernel is expected to be the slowest; run it and compare your own numbers.
On Warp's CPU device threads run one after another, there are no lanes, so the three times are
close: divergence is a GPU effect.
Run: python ex6_divergence.py [--dump-json PATH]
"""
import argparse
import json
import time

import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
N = (1 << 22) if ON_GPU else (1 << 16)            # elements; multiple of 8 * 32
M = N // 8                                         # elements per branch
ITERS = 64


@wp.func
def work(branch: int, x: float) -> float:
    r = x
    if branch == 0:
        for _ in range(ITERS):
            r = wp.sin(r) + 0.5
    elif branch == 1:
        for _ in range(ITERS):
            r = wp.cos(r) * 0.9
    elif branch == 2:
        for _ in range(ITERS):
            r = wp.sqrt(r * r + 1.0) - 0.9
    elif branch == 3:
        for _ in range(ITERS):
            r = wp.exp(-r * r)
    elif branch == 4:
        for _ in range(ITERS):
            r = wp.tanh(r) + 0.25
    elif branch == 5:
        for _ in range(ITERS):
            r = 1.0 / (1.0 + r * r)
    elif branch == 6:
        for _ in range(ITERS):
            r = wp.log(1.0 + r * r)
    else:
        for _ in range(ITERS):
            r = wp.atan(r) + 0.1
    return r


@wp.kernel
def interleaved(x: wp.array[float], out: wp.array[float]):
    i = wp.tid()
    out[i] = work(i % 8, x[i])                     # neighbours take different branches


@wp.kernel
def grouped(m: int, x: wp.array[float], out: wp.array[float]):
    i = wp.tid()
    out[i] = work(i // m, x[i])                    # a whole warp shares one branch


@wp.kernel
def one_branch(branch: int, m: int, x: wp.array[float], out: wp.array[float]):
    j = wp.tid()
    i = branch * m + j
    out[i] = work(branch, x[i])                    # every thread of this launch: same branch


x_np = np.random.default_rng(0).random(N, dtype=np.float32)   # grouped order: branch-major
x = wp.array(x_np)
# the same values in interleaved order: element 8 * j + c holds x_np[c * M + j]
x_inter = wp.array(x_np.reshape(8, M).T.copy().reshape(N))
out_a = wp.zeros(N, dtype=float)
out_b = wp.zeros(N, dtype=float)
out_c = wp.zeros(N, dtype=float)


def run_interleaved():
    wp.launch(interleaved, dim=N, inputs=[x_inter], outputs=[out_a])


def run_grouped():
    wp.launch(grouped, dim=N, inputs=[M, x], outputs=[out_b])


def run_specialized():
    for branch in range(8):
        wp.launch(one_branch, dim=M, inputs=[branch, M, x], outputs=[out_c])


def best_ms(fn, reps=5):
    fn()                                           # warm-up: JIT compile + first run
    wp.synchronize()
    best = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        wp.synchronize()                           # wait for the device before reading the clock
        best = min(best, time.perf_counter() - t0)
    return 1e3 * best


times = {"interleaved": best_ms(run_interleaved), "grouped": best_ms(run_grouped),
         "specialized": best_ms(run_specialized)}

# all three compute the same numbers (interleaved stores them in interleaved order)
a_back = out_a.numpy().reshape(M, 8).T.reshape(N)
print(f"device: {wp.get_device()} | N = {N} | {ITERS} iterations per element")
for k, v in times.items():
    print(f"  {k:12s} {v:8.2f} ms   ({v / times['grouped']:.2f}x grouped)")
print("same results in all three:",
      bool(np.array_equal(out_b.numpy(), out_c.numpy()) and np.array_equal(a_back, out_b.numpy())))
if not ON_GPU:
    print("CPU device: threads run in a loop, no 32-lane warps, so no divergence penalty.")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "n": N, "iters": ITERS,
                   "ms": {k: round(v, 2) for k, v in times.items()}}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
%%writefile ex7_coalescing.py
"""Example 4.7 - memory coalescing: which index do consecutive threads walk?

Adds two (nworld, n) arrays. Both kernels do identical work; only the mapping from thread
index to array element differs:
  coalesced  w, j = wp.tid() with dim=(nworld, n): neighbouring threads read neighbouring
             addresses (the contiguous last axis) -> a GPU warp needs few memory transactions
  strided    j, w = wp.tid() with dim=(n, nworld): neighbouring threads are a whole row
             (n * 4 bytes) apart -> one transaction per lane
This is why MJWarp puts nworld first and launches dim=(nworld, n_elements).
On the CPU device threads run in order, so the strided kernel jumps n * 4 bytes per thread
and loses the CPU cache instead: it is slower too, for a different reason.
Run: python ex7_coalescing.py [--dump-json PATH]
"""
import argparse
import json
import time

import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD, N = (8192, 4096) if ON_GPU else (2048, 2048)


@wp.kernel
def add_coalesced(a: wp.array2d[float], b: wp.array2d[float], c: wp.array2d[float]):
    w, j = wp.tid()                                # last launch axis = contiguous array axis
    c[w, j] = a[w, j] + b[w, j]


@wp.kernel
def add_strided(a: wp.array2d[float], b: wp.array2d[float], c: wp.array2d[float]):
    j, w = wp.tid()                                # last launch axis = the row index
    c[w, j] = a[w, j] + b[w, j]


rng = np.random.default_rng(0)
a = wp.array(rng.random((NWORLD, N), dtype=np.float32))
b = wp.array(rng.random((NWORLD, N), dtype=np.float32))
c1 = wp.zeros((NWORLD, N), dtype=float)
c2 = wp.zeros((NWORLD, N), dtype=float)


def best_ms(fn, reps=5):
    fn()                                           # warm-up (JIT)
    wp.synchronize()
    best = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        wp.synchronize()
        best = min(best, time.perf_counter() - t0)
    return 1e3 * best


t_co = best_ms(lambda: wp.launch(add_coalesced, dim=(NWORLD, N), inputs=[a, b], outputs=[c1]))
t_st = best_ms(lambda: wp.launch(add_strided, dim=(N, NWORLD), inputs=[a, b], outputs=[c2]))
gbytes = 3 * NWORLD * N * 4 / 1e9                  # 2 reads + 1 write of float32

print(f"device: {wp.get_device()} | array ({NWORLD}, {N}) float32, {gbytes:.3f} GB moved")
print(f"  coalesced {t_co:8.2f} ms  {gbytes / (t_co / 1e3):7.1f} GB/s")
print(f"  strided   {t_st:8.2f} ms  {gbytes / (t_st / 1e3):7.1f} GB/s"
      f"   ({t_st / t_co:.1f}x slower)")
print("same result:", bool(np.array_equal(c1.numpy(), c2.numpy())))
print("thread k reads address offset (floats): coalesced k, strided k *", N)
if not ON_GPU:
    print("CPU device: the gap comes from CPU caches, not from GPU memory transactions.")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "shape": [NWORLD, N],
                   "ms": {"coalesced": round(t_co, 2), "strided": round(t_st, 2)}},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Run the same scripts on the GPU and, in a fresh process with no visible GPU, on Warp's CPU device.
import os
import subprocess
import sys

if ON_GPU:
    for script in ["ex2_saxpy.py", "ex6_divergence.py", "ex7_coalescing.py"]:
        for label, extra in [("GPU", {}), ("CPU device", {"CUDA_VISIBLE_DEVICES": ""})]:
            res = subprocess.run([sys.executable, script], env={**os.environ, **extra},
                                 capture_output=True, text=True)
            print(f"--- {script} on {label} (own problem size, see first line; compare ratios, not ms)")
            print(res.stdout.strip() or res.stderr.strip()[-500:])
else:
    print("skipped on CPU: this comparison needs a GPU runtime (the scripts above already ran on CPU)")

## 5. Device functions
`@wp.func` helpers run on the device, can return values and are inlined. This example reuses MJWarp's private `math.rot_vec_quat` (the name may change between versions).

In [ ]:
# Example 4.4 - ex4_func.py
"""Example 4.4 - @wp.func device functions, reusing MJWarp's own math helpers.

A kernel calls our @wp.func, which calls mujoco_warp._src.math.rot_vec_quat (private API) to
rotate a vec3 by a MuJoCo-ordered (w, x, y, z) quaternion. The result is compared with
mujoco.mju_rotVecQuat, MuJoCo's C function.
Run: python ex4_func.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw  # noqa: F401  (imported for the course-wide import order)
from mujoco_warp._src import math as mjw_math   # private API: names may change

np.set_printoptions(precision=4, suppress=True)


@wp.func
def rotate_and_scale(v: wp.vec3, q: wp.quat, s: float) -> wp.vec3:
    """A device function: called from kernels, inlined by the compiler, may return a value."""
    return s * mjw_math.rot_vec_quat(v, q)


@wp.kernel
def rotate_all(vec_in: wp.array[wp.vec3], quat_in: wp.array[wp.quat],
               vec_out: wp.array[wp.vec3]):
    i = wp.tid()
    vec_out[i] = rotate_and_scale(vec_in[i], quat_in[i], 1.0)


rng = np.random.default_rng(1)
N = 4
quats = rng.standard_normal((N, 4))
quats /= np.linalg.norm(quats, axis=1, keepdims=True)        # unit quats, MuJoCo order (w,x,y,z)
vecs = rng.standard_normal((N, 3))

out = wp.zeros(N, dtype=wp.vec3)
wp.launch(rotate_all, dim=N,
          inputs=[wp.array(vecs, dtype=wp.vec3), wp.array(quats, dtype=wp.quat)],
          outputs=[out])

ref = np.zeros((N, 3))
for i in range(N):
    mujoco.mju_rotVecQuat(ref[i], vecs[i], quats[i])          # C MuJoCo, float64

print("quat[0] (w, x, y, z):", quats[0])
print("MJWarp rot_vec_quat :", out.numpy()[0])
print("mujoco.mju_rotVecQuat:", ref[0])
print("max abs difference over", N, "vectors:", f"{np.max(np.abs(out.numpy() - ref)):.1e}",
      "(float32 vs float64)")

## 6. The `wp.quat` storage pitfall
MJWarp stores (w, x, y, z) in `wp.quat` slots; Warp's built-ins such as `wp.quat_rotate` read (x, y, z, w).

In [ ]:
# Example 4.5 - ex5_quat_pitfall.py
"""Example 4.5 - the quaternion storage pitfall: wp.quat_rotate expects (x, y, z, w).

MuJoCo and MJWarp store quaternions as (w, x, y, z) inside wp.quat slots. Warp's own built-ins
(wp.quat_rotate, wp.transform, ...) read the same four slots as (x, y, z, w). Passing a
MuJoCo-ordered quaternion straight to wp.quat_rotate silently gives a different rotation.
Run: python ex5_quat_pitfall.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw  # noqa: F401  (imported for the course-wide import order)
from mujoco_warp._src import math as mjw_math   # private API: names may change

np.set_printoptions(precision=4, suppress=True)


@wp.kernel
def three_ways(v: wp.vec3, q_mj: wp.quat, out: wp.array[wp.vec3]):
    # q_mj holds MuJoCo order: q_mj[0] = w, q_mj[1..3] = x, y, z
    out[0] = mjw_math.rot_vec_quat(v, q_mj)                  # MJWarp helper: (w, x, y, z)
    out[1] = wp.quat_rotate(q_mj, v)                         # WRONG: Warp reads (x, y, z, w)
    q_warp = wp.quat(q_mj[1], q_mj[2], q_mj[3], q_mj[0])     # reorder to Warp's (x, y, z, w)
    out[2] = wp.quat_rotate(q_warp, v)                       # right again


# 90 degrees about z, in MuJoCo order (w, x, y, z)
q_mj = np.array([np.cos(np.pi / 4), 0.0, 0.0, np.sin(np.pi / 4)])
v = np.array([1.0, 0.0, 0.0])

out = wp.zeros(3, dtype=wp.vec3)
wp.launch(three_ways, dim=1, inputs=[wp.vec3(*v), wp.quat(*q_mj)], outputs=[out])
ref = np.zeros(3)
mujoco.mju_rotVecQuat(ref, v, q_mj)

r = np.round(out.numpy(), 6) + 0.0                         # tidy -0.0 and 1e-8 noise
print("rotate (1, 0, 0) by 90 deg about z; quat (w, x, y, z) =", q_mj)
print("mujoco.mju_rotVecQuat            :", ref)
print("mjw math.rot_vec_quat (w,x,y,z)  :", r[0])
print("wp.quat_rotate, MuJoCo order     :", r[1], "<- wrong")
print("wp.quat_rotate, reordered x,y,z,w:", r[2])
print("wp.quat_identity() slots         :", np.array(wp.quat_identity()), "(w is LAST in Warp)")

## 7. Compile-time specialization: `wp.static`
A kernel factory produces one specialized module per flag value.

In [ ]:
# Example 4.8 - ex8_static.py
"""Example 4.8 - compile-time specialization with wp.static inside a kernel factory.

make_clip(use_abs) returns a kernel in which `if wp.static(use_abs):` is decided while Warp
generates code, so each specialization contains only one side of the branch. The same pattern
appears all over MJWarp (factories are cached by warp_util.cache_kernel; Ch 26).
Run: python ex8_static.py
"""
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

np.set_printoptions(precision=4, suppress=True)

POWER = 3                                          # a Python int, read at compile time


def make_clip(use_abs: bool):
    @wp.kernel(module="unique", enable_backward=False)
    def clip(x_in: wp.array[float], y_out: wp.array[float]):
        i = wp.tid()
        v = x_in[i]
        if wp.static(use_abs):                     # resolved during code generation
            v = wp.abs(v)
        r = float(1.0)
        for _ in range(wp.static(POWER)):          # constant trip count: unrolled
            r *= v
        y_out[i] = wp.clamp(r, -1.0, 1.0)
    return clip


x = wp.array(np.array([-1.5, -0.5, 0.5, 1.5], dtype=np.float32))
y = wp.zeros(4, dtype=float)
kernels = {flag: make_clip(flag) for flag in (False, True)}
for flag, k in kernels.items():
    wp.launch(k, dim=4, inputs=[x], outputs=[y])
    expr = "clamp(|x|**3)" if flag else "clamp(x**3)  "
    print(f"use_abs={flag!s:5s} -> {expr} =", y.numpy(), "| module", k.module.name)

again = make_clip(True)                            # same source + same static values
print("modules compiled:", len({k.module.name for k in kernels.values()}))
print("make_clip(True) twice -> same kernel object:", again is kernels[True])

## 8. JIT compilation and the kernel cache
The only cell in the course that turns Warp's INFO log back on. In a notebook Warp was initialized by the setup cell, so no banner is captured; on a fresh Colab runtime the module lines say `(compiled)`, rerun the cell after a restart to see `(cached)`.

In [ ]:
# Example 4.12 - ex12_jit_log.py
"""Example 4.12 - JIT compilation, the kernel cache and Warp's INFO log.

This is the one script in the course that leaves Warp at its default log level (INFO) on
purpose: Warp then prints one "Module ... load on device ... took ... ms (compiled|cached)"
line per module it loads. A first mjw.step loads dozens of modules. We collect those lines with
wp.set_logger, then switch to wp.LOG_WARNING (what every other example does right after
`import warp`) and load a brand-new module: no line at all, although it is still compiled on a
cold cache, or loaded from the kernel cache on a warm one.
Run: python ex12_jit_log.py   (run it twice: the second run says "cached")
"""
import re
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_INFO                  # Warp 1.19 default, shown once in the course
import mujoco_warp as mjw

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""


class CollectingLogger:
    """Implements Warp's Logger protocol (debug/info/warning/error) and keeps the messages."""

    def __init__(self):
        self.lines = []

    def debug(self, message):
        self.lines.append(message)

    def info(self, message):
        self.lines.append(message)

    def warning(self, message, category=None, stacklevel=1):
        self.lines.append(message)

    def error(self, message):
        self.lines.append(message)


def module_lines(log):
    return [s for s in log.lines if s.startswith("Module ")]


@wp.kernel(module="unique")
def double_it(x: wp.array[float]):
    i = wp.tid()
    x[i] = 2.0 * x[i]


# --- phase 1: default INFO level -------------------------------------------------------
log1 = CollectingLogger()
wp.set_logger(log1)
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2, nconmax=4, njmax=16)
t0 = time.perf_counter()
mjw.step(m, d)                                     # first step: loads every module it needs
wp.synchronize()
t_first = time.perf_counter() - t0
t0 = time.perf_counter()
mjw.step(m, d)                                     # second step: modules already in memory
wp.synchronize()
t_second = time.perf_counter() - t0
lines1 = module_lines(log1)

# --- phase 2: the course setting ---------------------------------------------------------
wp.config.log_level = wp.LOG_WARNING
log2 = CollectingLogger()
wp.set_logger(log2)
x = wp.ones(4, dtype=float)
t0 = time.perf_counter()
wp.launch(double_it, dim=4, inputs=[x])            # a new module: compiled or loaded silently
t_new = time.perf_counter() - t0
lines2 = module_lines(log2)
wp.set_logger(None)                                # restore Warp's default logger

banner = [s for s in log1.lines if s.startswith("Warp ")]
first = banner[0].splitlines()[0] if banner else "none (Warp was already initialized)"
print("INFO level   : init banner captured:", first)
print("INFO level   : 'Module' lines during the first mjw.step:", len(lines1))
print("  of which (cached):", sum("(cached)" in s for s in lines1),
      "| (compiled):", sum("(compiled)" in s for s in lines1))
if lines1:
    print("  example line:", re.sub(r"took [0-9.]+ ms", "took ... ms", lines1[0]))
print(f"  first step {1e3 * t_first:.0f} ms, second step {1e3 * t_second:.1f} ms")
print("WARNING level: 'Module' lines for a new module:", len(lines2),
      f"(it still took {1e3 * t_new:.0f} ms)", "| x =", x.numpy())
print("kernel cache directory:", wp.config.kernel_cache_dir)

## 9. Atomics and order-dependent sums
On the GPU, run the cell twice: the atomic totals can differ in their last digits.

In [ ]:
# Example 4.9 - ex9_atomics.py
"""Example 4.9 - wp.atomic_add, and why a float sum depends on the order of additions.

1M float32 values are summed by 1M threads into one accumulator with wp.atomic_add. Float
addition is not associative, so the result depends on the order in which threads arrive:
on a GPU that order changes from run to run (low bits flicker); on Warp's CPU device threads
run in index order, so the two runs match. A tiny 8-number case shows order dependence exactly.
Full treatment of determinism: Ch 22.
Run: python ex9_atomics.py
"""
import itertools

import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

N = 1 << 20


@wp.kernel
def atomic_sum(x: wp.array[float], total: wp.array[float]):
    i = wp.tid()
    wp.atomic_add(total, 0, x[i])                  # read-modify-write, no lost updates


x_np = np.random.default_rng(0).random(N, dtype=np.float32)
x = wp.array(x_np)
runs = []
for _ in range(2):
    total = wp.zeros(1, dtype=float)
    wp.launch(atomic_sum, dim=N, inputs=[x], outputs=[total])
    runs.append(float(total.numpy()[0]))

exact = float(np.sum(x_np, dtype=np.float64))
pairwise = float(np.sum(x_np))                     # NumPy float32 uses pairwise summation
print(f"device: {wp.get_device()} | N = {N}")
print(f"float64 sum (reference)  : {exact:.4f}")
print(f"NumPy float32 (pairwise) : {pairwise:.4f}   error {pairwise - exact:+.4f}")
print(f"atomic_add run 1         : {runs[0]:.4f}   error {runs[0] - exact:+.4f}")
print(f"atomic_add run 2         : {runs[1]:.4f}   bitwise equal to run 1: {runs[0] == runs[1]}")

# the same 8 float32 numbers added in every possible order (40320 orders)
vals = np.array([1e8, 3.0, -1e8, 0.75, 7.0, 0.125, -5.0, 2.5], dtype=np.float32)
results = set()
for perm in itertools.permutations(range(8)):
    acc = np.float32(0.0)
    for k in perm:
        acc = np.float32(acc + vals[k])
    results.add(float(acc))
print("8 values:", vals.tolist())
print("exact sum: 8.375 | distinct float32 results over all 40320 orders:", len(results))
print("smallest / largest:", min(results), "/", max(results))

## 10. Timing
Launches are asynchronous on a GPU: stop the clock only after `wp.synchronize()`.

In [ ]:
# Example 4.10 - ex10_timing.py
"""Example 4.10 - timing device code: launches are asynchronous on a GPU.

wp.launch on a CUDA device only queues the kernel and returns at once. A timer that stops
right after the launch measures Python, not the GPU. Call wp.synchronize() (or use
wp.ScopedTimer(..., synchronize=True)) before reading the clock. On Warp's CPU device the
launch runs the kernel before returning, so both numbers are close.
Run: python ex10_timing.py
"""
import time

import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

ON_GPU = wp.get_device().is_cuda
N = (1 << 24) if ON_GPU else (1 << 16)
ITERS = 200


@wp.kernel
def busy(x: wp.array[float], y: wp.array[float]):
    i = wp.tid()
    v = x[i]
    for _ in range(ITERS):
        v = wp.sin(v) * 0.99 + 0.01
    y[i] = v


x = wp.array(np.random.default_rng(0).random(N, dtype=np.float32))
y = wp.zeros(N, dtype=float)
wp.launch(busy, dim=N, inputs=[x], outputs=[y])    # warm-up: compile + load the module
wp.synchronize()

t0 = time.perf_counter()
wp.launch(busy, dim=N, inputs=[x], outputs=[y])
t_no_sync = time.perf_counter() - t0               # WRONG on GPU: kernel may still be running
wp.synchronize()

t0 = time.perf_counter()
wp.launch(busy, dim=N, inputs=[x], outputs=[y])
wp.synchronize()                                   # wait until the device is idle
t_sync = time.perf_counter() - t0

with wp.ScopedTimer("busy", synchronize=True, print=False) as timer:
    wp.launch(busy, dim=N, inputs=[x], outputs=[y])

print(f"device: {wp.get_device()} | N = {N} | {ITERS} sin() per element")
print(f"  stop clock right after wp.launch : {1e3 * t_no_sync:8.2f} ms")
print(f"  stop clock after wp.synchronize(): {1e3 * t_sync:8.2f} ms")
print(f"  wp.ScopedTimer(synchronize=True) : {timer.elapsed:8.2f} ms")
if ON_GPU:
    print("GPU: the first number is only the cost of queueing the launch.")
else:
    print("CPU device: launches run synchronously, so the numbers are close.")

## 11. Going deeper: tiles preview
`wp.launch_tiled` gives each logical index a block of threads that cooperate on tiles (owner chapter: Ch 26).

In [ ]:
# Example 4.11 - ex11_tile_sum.py
"""Example 4.11 (going deeper) - a tiled per-world reduction with wp.launch_tiled.

launch_tiled starts block_dim threads per logical index (one block per world here). The block
cooperatively loads one world's row into a tile (shared memory on a GPU), reduces it with
wp.tile_sum and stores the result. On Warp's CPU device the block collapses to one thread
(unless wp.config.enable_cpu_blocks is set), and the code still runs. Owner chapter: Ch 26.
Run: python ex11_tile_sum.py
"""
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

NW, N = 4, 64
TILE = N                                           # tile shapes must be compile-time constants


@wp.kernel
def per_world_sum(data: wp.array2d[float], out: wp.array[float]):
    worldid = wp.tid()                             # one logical index per block
    t = wp.tile_load(data[worldid], shape=(TILE,))   # the whole row, loaded by the block
    s = wp.tile_sum(t)                             # cooperative reduction -> 1-element tile
    wp.tile_store(out, s, offset=(worldid,))


arr_np = np.arange(NW * N, dtype=np.float32).reshape(NW, N)
arr = wp.array(arr_np)
out = wp.zeros(NW, dtype=float)
wp.launch_tiled(per_world_sum, dim=[NW], inputs=[arr], outputs=[out], block_dim=32)

print("device:", wp.get_device(), "| block_dim requested: 32")
print("tile sums :", out.numpy().astype(int).tolist())
print("NumPy sums:", arr_np.sum(axis=1).astype(int).tolist())

## Exercise 4.1: clamp a batched control array
Fill in the kernel so the assert passes.

In [ ]:
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING


@wp.kernel
def clamp_ctrl(ctrl: wp.array2d[float]):
    worldid, actid = wp.tid()
    # TODO: clamp ctrl[worldid, actid] to [-1, 1]


NWORLD, NU = 8, 3
ctrl_np = np.random.default_rng(0).normal(0.0, 2.0, (NWORLD, NU)).astype(np.float32)
ctrl = wp.array(ctrl_np)
wp.launch(clamp_ctrl, dim=(NWORLD, NU), inputs=[ctrl])
assert np.allclose(ctrl.numpy(), np.clip(ctrl_np, -1.0, 1.0))
print("Exercise 4.1 passed")

In [ ]:
#@title Exercise 4.1: solution (double-click to show the code)
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING


@wp.kernel
def clamp_ctrl(ctrl: wp.array2d[float]):
    worldid, actid = wp.tid()
    ctrl[worldid, actid] = wp.clamp(ctrl[worldid, actid], -1.0, 1.0)


NWORLD, NU = 8, 3
ctrl_np = np.random.default_rng(0).normal(0.0, 2.0, (NWORLD, NU)).astype(np.float32)
ctrl = wp.array(ctrl_np)
wp.launch(clamp_ctrl, dim=(NWORLD, NU), inputs=[ctrl])
assert np.allclose(ctrl.numpy(), np.clip(ctrl_np, -1.0, 1.0))
print("Exercise 4.1 passed")

## Exercise 4.2: per-world kinetic energy of point masses
One thread per world sums 0.5 * m_i * |v_i|^2. Type the accumulator (`total = float(0.0)`).

In [ ]:
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING


@wp.kernel
def kinetic_energy(mass: wp.array[float], vel: wp.array2d[wp.vec3], ke_out: wp.array[float]):
    worldid = wp.tid()
    # TODO: loop over vel.shape[1] masses and write the sum to ke_out[worldid]


NWORLD, NP = 8, 5
rng = np.random.default_rng(1)
mass_np = rng.uniform(0.5, 2.0, NP).astype(np.float32)
vel_np = rng.normal(size=(NWORLD, NP, 3)).astype(np.float32)
ke = wp.zeros(NWORLD, dtype=float)
wp.launch(kinetic_energy, dim=NWORLD,
          inputs=[wp.array(mass_np), wp.array(vel_np, dtype=wp.vec3)], outputs=[ke])
ref = 0.5 * np.sum(mass_np[None, :] * np.sum(vel_np**2, axis=2), axis=1)
assert np.allclose(ke.numpy(), ref, rtol=1e-5)
print("Exercise 4.2 passed")

In [ ]:
#@title Exercise 4.2: solution (double-click to show the code)
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING


@wp.kernel
def kinetic_energy(mass: wp.array[float], vel: wp.array2d[wp.vec3], ke_out: wp.array[float]):
    worldid = wp.tid()
    total = float(0.0)
    for i in range(vel.shape[1]):
        v = vel[worldid, i]
        total += 0.5 * mass[i] * wp.dot(v, v)
    ke_out[worldid] = total


NWORLD, NP = 8, 5
rng = np.random.default_rng(1)
mass_np = rng.uniform(0.5, 2.0, NP).astype(np.float32)
vel_np = rng.normal(size=(NWORLD, NP, 3)).astype(np.float32)
ke = wp.zeros(NWORLD, dtype=float)
wp.launch(kinetic_energy, dim=NWORLD,
          inputs=[wp.array(mass_np), wp.array(vel_np, dtype=wp.vec3)], outputs=[ke])
ref = 0.5 * np.sum(mass_np[None, :] * np.sum(vel_np**2, axis=2), axis=1)
assert np.allclose(ke.numpy(), ref, rtol=1e-5)
print("Exercise 4.2 passed")

## Stretch: a tiled per-world maximum
Change Example 4.11 to compute each world's maximum. Then, on the GPU, time it against a one-thread-per-world loop for NW = 4096 and rows of 64 and 1024 floats.

In [ ]:
# your code here

In [ ]:
#@title Stretch: solution (double-click to show the code)
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

NW, TILE = 4, 64


@wp.kernel
def per_world_max(data: wp.array2d[float], out: wp.array[float]):
    worldid = wp.tid()
    t = wp.tile_load(data[worldid], shape=(TILE,))
    wp.tile_store(out, wp.tile_max(t), offset=(worldid,))


arr_np = np.random.default_rng(2).normal(size=(NW, TILE)).astype(np.float32)
out = wp.zeros(NW, dtype=float)
wp.launch_tiled(per_world_max, dim=[NW], inputs=[wp.array(arr_np)], outputs=[out], block_dim=32)
assert np.allclose(out.numpy(), arr_np.max(axis=1))
print("Stretch passed:", out.numpy())